In [4]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.lines import Line2D
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
import plotly.colors as pc
import os

In [5]:
xvals = []
for i in range(0,101,1):
    xvals.append(i/9269)
for i in range(200,9300,100):
    xvals.append(i/9269)
for i in range(1,41,1):
    xvals.append(i)
    
x_first_ep = []
for i in range(0,101,1):
    x_first_ep.append(i/9269)
for i in range(200,9300,100):
    x_first_ep.append(i/9269)
    
x_100_first_batches = []
for i in range(0,101,1):
    x_100_first_batches.append(i/9269)

In [6]:
def compose_weight_tensor(BASE_PATH, arg1, arg2, arg3 ):
    layers = [1,2]
    gates = ['forget', 'input', 'output', 'cell']
    weight_types = ['ih', 'hh']
    checkpoints = {}
    for layer in layers:
        for gate in gates:
            for weight_type in weight_types:
                check = []
                check.append(torch.load(f'{BASE_PATH}/layer{layer}_{gate}_gate_{weight_type}_{arg1}.pt'))
                check.append(torch.load(f'{BASE_PATH}/layer{layer}_{gate}_gate_{weight_type}_{arg2}.pt'))
                check.append(torch.load(f'{BASE_PATH}/layer{layer}_{gate}_gate_{weight_type}_{arg3}.pt'))
                check = torch.cat(check, dim=0)
                checkpoints[f'layer{layer}_{gate}_gate_{weight_type}'] = check
    return checkpoints
            

In [7]:
BASE_PATH = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check/weights'
checkpoints = compose_weight_tensor(
    BASE_PATH,
    'first100batches',
    'every100batches',
    'ep'
)

In [8]:
def get_sv_per_gate(checkpoint, gates, layer,num_checkpoints):

    
    sv_per_gate = {}
    for gate in gates :
    # For each checkpoint
        all_singular_values = []
        for i in range(num_checkpoints):
            
            # Combine all weights horizontally (concatenate along input dimension)
            #  ih is [hidden_size, input_size] and hh is [hidden_size, hidden_size] (https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
            all_weights = torch.cat([checkpoint[f'layer{layer}_{gate}_gate_ih'][i], checkpoint[f'layer{layer}_{gate}_gate_hh'][i]], dim=1)
            
            # Compute singular values
            u, s, v = torch.svd(all_weights)
            
            # Store singular values for this checkpoint
            all_singular_values.append(s)
            
        sv_per_gate[f'{gate}']= torch.stack(all_singular_values, dim=0)
        
    return sv_per_gate

In [9]:
gates = ['forget', 'cell', 'input', 'output']
l1_sv_per_gate = get_sv_per_gate(checkpoints, gates, 1, checkpoints['layer1_forget_gate_ih'].shape[0] )

In [10]:
l2_sv_per_gate = get_sv_per_gate(checkpoints, gates, 2, checkpoints['layer1_forget_gate_ih'].shape[0])

In [11]:

def plot_singular_values(all_singular_values, xvals, layer_name="Layer 1"):
    """
    Plots the singular values of a layer's weights through time (checkpoints) using Plotly
    with a gradient color representing the order of singular values.

    Args:
        all_singular_values (list of numpy.ndarray): List of singular values for each checkpoint.
        layer_name (str): Name of the layer to include in the plot title.
    """
    num_checkpoints = len(all_singular_values)
    num_singular_values = max(len(sv) for sv in all_singular_values)

    # Normalize singular value index for color mapping
    colorscale = pc.get_colorscale("Viridis")  # Choose any Plotly colormap
    color_values = np.linspace(0, 1, num_singular_values)
    color_mapper = pc.sample_colorscale(colorscale, color_values, colortype='rgb')

    fig = go.Figure()

    for i in range(num_singular_values):
        singular_value_through_time = [sv[i] if i < len(sv) else np.nan for sv in all_singular_values]
        fig.add_trace(go.Scatter(
            x=xvals,
            y=singular_value_through_time,
            mode='lines',
            line=dict(color='blue'),
            name=f"SV {i+1}",
            showlegend=False  # Hide individual legend items
        ))

    fig.update_layout(
        title=f'{layer_name}',
        xaxis_title='Checkpoint',
        yaxis_title='Singular Value',
        hovermode='closest'
    )

    fig.show()


In [ ]:
#Forget gate 
#Layer 1
plot_singular_values(l1_sv_per_gate['forget'], xvals, layer_name="SVs of Forget Gate in Layer 1 (40 epochs)")
plot_singular_values(l1_sv_per_gate['forget'], x_first_ep, layer_name="SVs of Forget Gate in Layer 1 (1st epoch)")
plot_singular_values(l1_sv_per_gate['forget'], x_100_first_batches, layer_name="SVs of Forget Gate in Layer 1 (1st 100 batches)")

#Layer 2
plot_singular_values(l2_sv_per_gate['forget'], xvals, layer_name="SVs of Forget Gate in Layer 2 (40 epochs)")
plot_singular_values(l2_sv_per_gate['forget'], x_first_ep, layer_name="SVs of Forget Gate in Layer 2 (1st epoch)")
plot_singular_values(l2_sv_per_gate['forget'], x_100_first_batches, layer_name="SVs of Forget Gate in Layer 2 (1st 100 batches)")



In [ ]:
#cell gate 
#Layer 1
plot_singular_values(l1_sv_per_gate['cell'], xvals, layer_name="SVs of Cell State in Layer 1 (40 epochs)")
plot_singular_values(l1_sv_per_gate['cell'], x_first_ep, layer_name="SVs of Cell State in Layer 1 (1st epoch)")
plot_singular_values(l1_sv_per_gate['cell'], x_100_first_batches, layer_name="SVs of Cell State in Layer 1 (1st 100 batches)")

#Layer 2
plot_singular_values(l2_sv_per_gate['cell'], xvals, layer_name="SVs of Cell State in Layer 2 (40 epochs)")
plot_singular_values(l2_sv_per_gate['cell'], x_first_ep, layer_name="SVs of Cell State in Layer 2 (1st epoch)")
plot_singular_values(l2_sv_per_gate['cell'], x_100_first_batches, layer_name="SVs of Cell State in Layer 2 (1st 100 batches)")



In [ ]:
#input gate 
#Layer 1
plot_singular_values(l1_sv_per_gate['input'], xvals, layer_name="SVs of Input Gate in Layer 1 (40 epochs)")
plot_singular_values(l1_sv_per_gate['input'], x_first_ep, layer_name="SVs of Input Gate in Layer 1 (1st epoch)")
plot_singular_values(l1_sv_per_gate['input'], x_100_first_batches, layer_name="SVs of Input Gate in Layer 1 (1st 100 batches)")

#Layer 2
plot_singular_values(l2_sv_per_gate['input'], xvals, layer_name="SVs of Input Gate in Layer 2 (40 epochs)")
plot_singular_values(l2_sv_per_gate['input'], x_first_ep, layer_name="SVs of Input Gate in Layer 2 (1st epoch)")
plot_singular_values(l2_sv_per_gate['input'], x_100_first_batches, layer_name="SVs of Input Gate in Layer 2 (1st 100 batches)")



In [ ]:
#output gate 
#Layer 1
plot_singular_values(l1_sv_per_gate['output'], xvals, layer_name="SVs of Output Gate in Layer 1 (40 epochs)")
plot_singular_values(l1_sv_per_gate['output'], x_first_ep, layer_name="SVs of Output Gate in Layer 1 (1st epoch)")
plot_singular_values(l1_sv_per_gate['output'], x_100_first_batches, layer_name="SVs of Output Gate in Layer 1 (1st 100 batches)")

#Layer 2
plot_singular_values(l2_sv_per_gate['output'], xvals, layer_name="SVs of Output Gate in Layer 2 (40 epochs)")
plot_singular_values(l2_sv_per_gate['output'], x_first_ep, layer_name="SVs of Output Gate in Layer 2 (1st epoch)")
plot_singular_values(l2_sv_per_gate['output'], x_100_first_batches, layer_name="SVs of Output Gate in Layer 2 (1st 100 batches)")
